## Run test: zai-org/GLM-4.1V-9B-Thinking

Aligned with the Hugging Face README Quick Inference example. Inference Providers snippets are ignored because they do not validate local AMD GPU loading. The direct path uses Glm4vForConditionalGeneration, matching the README.

In [ ]:
%pip install -U transformers accelerate

In [ ]:
# HF Colab aligned image-text-to-text pipeline smoke test on one visible GPU.
import gc
import torch
from PIL import Image, ImageDraw
from transformers import pipeline

model_path = "/disk/ssd1/zihaomu_amd/models/zai-org__GLM-4.1V-9B-Thinking"
img_path = "/tmp/hf_validation_black_square.png"
img = Image.new("RGB", (224, 224), "white")
draw = ImageDraw.Draw(img)
draw.rectangle([50, 50, 170, 170], fill="black")
img.save(img_path)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": img_path},
            {"type": "text", "text": "What color is the square in the image?"},
        ],
    },
]
pipe = pipeline("image-text-to-text", model=model_path, device=0, trust_remote_code=True)
pipe_result = pipe(messages, max_new_tokens=40)
print(pipe_result)
pipe_tensor_devices = sorted({str(p.device) for p in pipe.model.parameters()})
print("pipe parameter devices =", pipe_tensor_devices)

del pipe
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# HF Colab aligned direct image inference smoke test.
import torch
from transformers import AutoProcessor, Glm4vForConditionalGeneration

processor = AutoProcessor.from_pretrained(model_path, use_fast=True)
model = Glm4vForConditionalGeneration.from_pretrained(model_path).to("cuda").eval()

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": img_path},
            {"type": "text", "text": "What color is the square in the image?"},
        ],
    },
]
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=40)
output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(output_text)
